In [0]:
from pyspark.sql import functions as F
from datetime import datetime , timezone
import uuid

In [0]:
CATALOG = "workspace"

SCHEMA = "retail_sales"

TARGET_SCHEMA = f"{CATALOG}.{SCHEMA}"

SOURCE_SCHEMA = "samples.tpcds_sf1"

print("Source:", SOURCE_SCHEMA)
print("Destination:", TARGET_SCHEMA)

In [0]:
source_df = spark.table(
    f"{SOURCE_SCHEMA}.store_sales"
)

display(source_df.limit(5))

In [0]:
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}"
)

print("Target schema is ready")

In [0]:
sales_source_df = (
    spark.table(f"{SOURCE_SCHEMA}.store_sales")
    .limit(10000)
)

display(sales_source_df.limit(10))

print(
    "Sales source rows:",
    sales_source_df.count()
)

In [0]:
sales_source_df.printSchema()

In [0]:
#Unique identifier for this pipeline execution
RUN_ID = str(uuid.uuid4())

#UTC time when this run started
INGESTED_AT = datetime.now(timezone.utc)

print("Run ID:", RUN_ID)
print("Ingestion time:", INGESTED_AT.isoformat())

In [0]:
def ingest_to_bronze(
    source_table,
    target_table,
    row_limit= None
):
    source_name = f"{SOURCE_SCHEMA}.{source_table}"
    target_name = f"{TARGET_SCHEMA}.{target_table}"

    print(f"\nReading: {source_name}")

    df = spark.table(source_name)

    if row_limit is not None:
        df = df.limit(row_limit)

    if len(df.columns) == 0:
        raise ValueError(
            f"Source table has no columns: {source_name}"
    )
        
    if "ingestion_run_id" in df.columns:
        raise ValueError("Reserved metada column exists")

    if "ingested_at" in df.columns:
        raise ValueError("Reserved timestamp column exists")

    bronze_df = (
        df
        .withColumn(
            "ingestion_run_id",
            F.lit(RUN_ID)
        )
        .withColumn(
            "ingested_at",
            F.lit(INGESTED_AT)
        )
        .withColumn(
            "source_table",
            F.lit(source_name)
        )
    )

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_name)
    )

    saved_df = spark.table(target_name)
    saved_count = saved_df.count()

    if saved_count == 0:
        raise ValueError(
            f"Empty target table:{target_name}"
            )
        
    print(f"Saved: {target_name}")
    print(f"Rows written: {saved_count}")

    return saved_count



    

In [0]:
SOURCE_TABLES = {
    "store_sales": "bronze_store_sales",
    "customer": "bronze_customers",
    "item": "bronze_items",
    "store": "bronze_stores",
    "date_dim": "bronze_dates"
}

print("Tables to ingest:")

for source, target in SOURCE_TABLES.items():
    print(f"{source} -> {target}")

In [0]:
ROW_LIMITS = {
    "store_sales": 10000,
    "customer": 10000,
    "item": 10000,
    "store": 1000,
    "date_dim": 100000
}

ingestion_results = []

for source_table, target_table in SOURCE_TABLES.items():

    count = ingest_to_bronze(
        source_table= source_table,
        target_table= target_table,
        row_limit = ROW_LIMITS[source_table]
    )

    ingestion_results.append({
        "source_table": source_table,
        "target_table": target_table,
        "row_count": count,
        "run_id": RUN_ID
    })

print("\nAll Bronze ingestion tasks completed!")

In [0]:
results_df = spark.createDataFrame(
    ingestion_results
)

display(results_df)

In [0]:
display(
    spark.sql(
        f"SHOW TABLES IN {TARGET_SCHEMA}"
    )
)

In [0]:
bronze_sales_df = spark.table(
    f"{TARGET_SCHEMA}.bronze_store_sales"
)

display(
    bronze_sales_df.select(
        "ss_item_sk",
        "ss_customer_sk",
        "ss_quantity",
        "ss_net_paid",
        "ingestion_run_id",
        "ingested_at",
        "source_table"
    ).limit(10)
)

In [0]:
important_columns = [
    "ss_item_sk",
    "ss_sold_date_sk",
    "ss_quantity",
    "ss_net_paid"
]

null_checks = [
    F.sum(
        F.col(column_name).isNull().cast("int")
    ).alias(column_name)
    for column_name in important_columns
]

display(
    bronze_sales_df.select(null_checks)
)

In [0]:
metadata_check_df = bronze_sales_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.col("ingested_at").isNull().cast("int")
    ).alias("missing_ingestion_timestamps"),

    F.sum(
        F.col("ingestion_run_id").isNull().cast("int")
    ).alias("missing_run_ids"),

    F.countDistinct("ingestion_run_id").alias(
        "distinct_run_ids"
    )
)

display(metadata_check_df)

In [0]:
validation = metadata_check_df.first()

assert validation["total_rows"] > 0, (
    "Bronze table is empty"
)

assert validation["missing_ingestion_timestamps"] == 0, (
    "Missing ingestion timestamps found"
)

assert validation["missing_run_ids"] == 0, (
    "Missing ingestion run IDs found"
)

assert validation["distinct_run_ids"] == 1, (
    "Unexpected number of ingestion runs"
)

print("All Bronze metadata checks passed!")

In [0]:
target_name = (
    f"{TARGET_SCHEMA}.bronze_store_sales"
)

count_before = spark.table(target_name).count()

print("Before rerun:", count_before)

In [0]:
new_count = ingest_to_bronze(
    source_table = "store_sales",
    target_table="bronze_store_sales",
    row_limit=10000
)

count_after = spark.table(target_name).count()

print("Before:", count_before)
print("After:", count_after)

assert count_before == count_after

print("Overwrite row-count check passed!")

In [0]:
display(
    spark.sql(
        f"DESCRIBE DETAIL {target_name}"
    )
)

In [0]:
display(
    spark.sql(
        f"DESCRIBE HISTORY {target_name}"
    )
)